# 第1章 环境准备 - 操作手册

**Goal**: 验证 ROCm 环境、PyTorch ROCm 后端和 HIP 编译工具链是否可用。

**Prerequisite**: 已安装 ROCm 7.13、uv 环境管理器、AMD GPU（本手册以 gfx1201 为基线）。

**Platform**: 原生 Ubuntu 24.04（推荐）或 WSL2（部分工具不可用）。

**参考文档**: `docs/part0-intro/chapter1/index.md`

**代码目录**: `code/part0-intro/chapter1/`

## 1. 定位仓库根目录

所有路径使用绝对路径，从仓库根目录开始。

## 2. 检测 GPU 架构

**Parameter**: 无

**Execution**: 运行 `rocminfo` 检测当前 GPU 架构。

**Expected output**: 输出检测到的架构（gfx1100/gfx1151/gfx1201）。

**Pass criteria**: 成功检测到支持的架构之一。

In [ ]:
# 检测当前 GPU 架构
import subprocess

rocminfo_result = subprocess.run(
    ["rocminfo"],
    capture_output=True,
    text=True,
    check=True
)

# 从 rocminfo 输出中提取架构
arch = "gfx1201"  # 默认值（gfx1201 为叙述基线）
if "gfx1100" in rocminfo_result.stdout:
    arch = "gfx1100"
elif "gfx1151" in rocminfo_result.stdout:
    arch = "gfx1151"
elif "gfx1201" in rocminfo_result.stdout:
    arch = "gfx1201"
else:
    raise RuntimeError("未检测到支持的架构 (gfx1100/gfx1151/gfx1201)")

print(f"检测到架构: {arch}")


In [ ]:
import pathlib
import subprocess

def find_repo_root():
    """搜索当前目录及父目录，定位包含 README.md + code/docs/notebooks 的仓库根目录。"""
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

## 3. 验证 ROCm 可见性

**Parameter**: 无

**Execution**: 运行 `rocminfo` 检查 GPU 设备。

**Expected output**: 输出包含 GPU 设备名称和 gfx 架构（如 gfx1201）。

**Pass criteria**: `rocminfo` 成功执行且输出包含 GPU 信息，status PASS。

In [ ]:
result = subprocess.run(["rocminfo"], capture_output=True, text=True, check=True)
print(result.stdout[:1000])  # 显示前1000字符
if "gfx" in result.stdout.lower():
    print("\nstatus PASS: ROCm 可见 GPU 设备")
else:
    print("\nstatus FAIL: 未检测到 GPU")

## 4. 验证 hipcc 编译器

**Parameter**: 无

**Execution**: 运行 `hipcc --version` 检查编译器版本。

**Expected output**: 输出 HIP 版本信息（如 7.13.99004）。

**Pass criteria**: `hipcc --version` 成功执行，status PASS。

In [ ]:
result = subprocess.run(["hipcc", "--version"], capture_output=True, text=True, check=True)
print(result.stdout)
print("status PASS: hipcc 可用")

## 5. 验证 PyTorch ROCm 后端

**Parameter**: 无

**Execution**: 运行 `check_torch_rocm.py` 脚本。

**Expected output**: 输出 PyTorch 版本、CUDA 可用性、设备名称和矩阵乘法结果。

**Pass criteria**: `torch.cuda.is_available()` 返回 True，矩阵乘法在 GPU 上成功执行，status PASS。

In [ ]:
check_torch_script = REPO_ROOT / "code/part0-intro/chapter1/check_torch_rocm.py"
result = subprocess.run(
    ["python", str(check_torch_script)],
    capture_output=True,
    text=True,
    check=True,
    cwd=REPO_ROOT / "code/part0-intro"
)
print(result.stdout)
if "cuda_available: True" in result.stdout:
    print("\nstatus PASS: PyTorch ROCm 后端可用")
else:
    print("\nstatus FAIL: PyTorch ROCm 后端不可用")

## 6. 编译并运行最小 HIP 程序

**Parameter**: 
- 源文件: `code/part0-intro/chapter1/vector_add.hip`
- 编译选项: `-O2`
- 目标架构: gfx1201（根据实际 GPU 调整）

**Execution**: 
1. 使用 `hipcc` 编译 `vector_add.hip`
2. 运行编译后的可执行文件

**Expected output**: 
- 编译成功，无错误
- 运行输出包含设备名称、向量大小、blocks、threads_per_block
- max_error 为 0
- status: PASS

**Pass criteria**: 编译成功，运行输出 `status: PASS`，max_error 为 0。

**Platform-specific commands**:
- gfx1100: `hipcc --offload-arch=gfx1100 -O2 vector_add.hip -o vector_add`
- gfx1151: `hipcc --offload-arch=gfx1151 -O2 vector_add.hip -o vector_add`
- gfx1201: `hipcc --offload-arch=gfx1201 -O2 vector_add.hip -o vector_add`

In [ ]:
import os

chapter1_dir = REPO_ROOT / "code/part0-intro/chapter1"
vector_add_hip = chapter1_dir / "vector_add.hip"
vector_add_bin = chapter1_dir / "vector_add"

# 编译（以 gfx1201 为基线）
compile_result = subprocess.run(
    ["hipcc", f"--offload-arch={arch}", "-O2", str(vector_add_hip), "-o", str(vector_add_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter1_dir
)
print("编译成功")
print(compile_result.stderr if compile_result.stderr else "无编译警告")

# 运行
run_result = subprocess.run(
    [str(vector_add_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter1_dir
)
print("\n运行结果:")
print(run_result.stdout)

if "status: PASS" in run_result.stdout:
    print("\n✓ 最小 HIP 程序验证通过")
else:
    print("\n✗ 最小 HIP 程序验证失败")

## 总结

本章完成了三道环境验证门：
1. ✓ ROCm 能看到 GPU（rocminfo）
2. ✓ PyTorch ROCm 后端可用（check_torch_rocm）
3. ✓ 最小 HIP 程序编译运行（vector_add）

所有验证通过后，环境已就绪，可以继续后续章节。